# 1. EDA and preprocessing
This challenge is entity resolution: each S1 row maps to zero or more S2/S3 IDs. The notebook profiles missingness, countries, duplicate fields, and the target cardinality using only supplied data.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'dataset').exists():
    REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'code/business_entity_resolution'))
from src.entity_resolution import prepare_frame

TRAIN = REPO_ROOT / 'dataset/train'
MAX_ROWS = 20_000  # set to None for the complete data
def read(name):
    return pd.read_csv(TRAIN / name, sep='\t', nrows=MAX_ROWS)

In [ ]:
s1 = read('train_source1.tsv')
s2 = read('train_source2.tsv')
s3 = read('train_source3.tsv')
truth = read('train_ground_truth.tsv')
for label, frame in {'s1': s1, 's2': s2, 's3': s3}.items():
    print(label, frame.shape)
    display(frame.isna().mean().rename('missing_fraction').to_frame())
    display(frame['country'].value_counts(dropna=False).head(20).to_frame('rows'))
truth['match_count'] = truth['matched_entity_ids'].fillna('').map(lambda x: 0 if not x else len(x.split(',')))
display(truth['match_count'].describe())
display(truth['match_count'].value_counts().sort_index().head(20).to_frame('s1_entities'))

In [ ]:
prepared = {name: prepare_frame(frame) for name, frame in {'s1': s1, 's2': s2, 's3': s3}.items()}
for name, frame in prepared.items():
    print(name, 'exact normalized names:', frame['name_key'].nunique(), '/', len(frame))
    print(name, 'empty names:', int((frame['name_norm'] == '').sum()), 'empty addresses:', int((frame['address_norm'] == '').sum()))

# Duplicate identifiers in the labelled data indicate a leakage or parsing problem.
print('duplicate entity IDs:', sum(frame['entity_id'].duplicated().sum() for frame in prepared.values()))
print('ground-truth S1 coverage:', truth['source1_entity_id'].isin(s1['entity_id']).mean())